# Debug from the trace; watch runs and sessions

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 23 §23.4–23.5 · type: walkthrough*

**One-line promise:** debug an agent failure you *cannot* reproduce by reading the trace, end the investigation by adding an eval case, then aggregate run- and **session**-level metrics and route signals correctly (page vs ticket).


## 🧠 Why this matters

Debugging an agent **inverts** the classic loop. Normally you reproduce first, then fix. Here you usually *can't* reproduce — the model sampled differently, retrieval changed, the world moved — so you investigate from **evidence**: the trace. The wrong final answer you observed is almost always *downstream* of the failure that matters (an empty retrieval, a tool error the model paraphrased into a "fact," a silently truncated context). Your job is to walk the span tree to the **first divergence point**, reproduce *that* at the right granularity, and then close the loop by turning the bug into a permanent eval case so the flywheel — not your memory — keeps it fixed.


## Objectives & prerequisites

**By the end you can:**
- Walk a span tree chronologically to the **first divergence point** and work the book's base-rate suspect list.
- Reproduce a *single* model call at the right granularity — resample 10× to tell a **tail event from the norm**.
- End a debugging session the right way: **add the failing case to the golden eval set** (links back to 22-03).
- Aggregate **operational** vs **quality** dashboard metrics, compute **session-level** metrics (turns-to-resolution, conversation success, abandonment, cost-per-resolved-conversation), and set **page-vs-ticket** alerting.

**Prereqs:** 23-01 (you need traces to debug and aggregate); 22-03 (the eval suite a fixed bug lands in). Runs fully offline on canned fixtures in `data/`.

**Extra packages beyond `requirements.txt`:** none — pure stdlib over committed JSON fixtures.


## Setup

Default `MOCK=1`: everything here operates on **canned traces** in `data/traces/` and offline aggregation of `data/sessions.json` — no collector, no network, no key. The only thing `MOCK=0` adds is an optional resample of a *single live* model call to show tail-vs-norm; the debugging method is identical either way.


In [ ]:
import json
import os
import random
import statistics
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
SEED = 23
random.seed(SEED)

if not MOCK and not os.getenv("ANTHROPIC_API_KEY"):
    raise SystemExit(
        "MOCK=0 but ANTHROPIC_API_KEY is not set. "
        "Set it in your .env, or unset COMPANION_MOCK to run offline."
    )

DATA = Path("data")
TRACES = DATA / "traces"

def load_trace(name):
    return json.loads((TRACES / name).read_text(encoding="utf-8"))

print(f"mode = {'MOCK (offline, free)' if MOCK else 'LIVE'} | seed = {SEED}")
print("fixtures:", [p.name for p in sorted(TRACES.glob('*.json'))])

### A tiny trace reader

The fixtures are span lists with `start_ms`/`end_ms` and parent ids — the same shape 23-01 produced, serialized. This helper prints any fixture as a chronological tree so we can read it like the chapter's `agent-trace` figure.


In [ ]:
def spans_in_order(trace):
    return sorted(trace["spans"], key=lambda s: s["start_ms"])

def print_trace(trace):
    by_parent = {}
    for s in trace["spans"]:
        by_parent.setdefault(s["parent_id"], []).append(s)
    for kids in by_parent.values():
        kids.sort(key=lambda s: s["start_ms"])
    def walk(span, depth):
        a = span.get("attributes", {})
        dur = span["end_ms"] - span["start_ms"]
        tag = "" if span["status"] == "OK" else f"  [{span['status']}]"
        extra = ""
        if "gen_ai.usage.input_tokens" in a:
            extra = f"  in={a['gen_ai.usage.input_tokens']} out={a['gen_ai.usage.output_tokens']}"
        if "retrieval.top_score" in a:
            extra = f"  top_score={a['retrieval.top_score']} chunks={a['retrieval.chunk_count']}"
        print("  " * depth + f"• {span['name']}  ({dur} ms){extra}{tag}")
        for c in by_parent.get(span["span_id"], []):
            walk(c, depth + 1)
    for root in by_parent.get(None, []):
        walk(root, 0)

t1 = load_trace("trace-retrieval-miss.json")
print("SYMPTOM: user reports the agent gave a refund window that isn't our policy.\n")
print_trace(t1)

### 🔮 Predict the divergence point from the symptom

The symptom: the agent stated a refund window that is **not** our policy. Before you read the spans' payloads —

*Predict:* which span is the **first divergence point** — the earliest step where reality left intent? Pick from the chapter's base-rate suspect list (in rough order): **bad retrieval → tool failed and model improvised → wrong rendered prompt → silent context truncation → model sampled badly**. Then run the next cell, which surfaces each span's evidence so you can locate it.


In [ ]:
SUSPECTS = [
    "bad retrieval (wrong/empty context)",
    "tool failed and the model improvised a 'fact'",
    "rendered prompt wasn't what you assumed",
    "silent context-window truncation",
    "model genuinely sampled badly",
]

def find_divergence(trace):
    """Walk chronologically; return the first span whose evidence implicates a suspect."""
    for s in spans_in_order(trace):
        a = s.get("attributes", {})
        if s["status"] == "ERROR":
            return s, SUSPECTS[1]
        if "retrieval.top_score" in a and a["retrieval.top_score"] < 0.35:
            return s, SUSPECTS[0]
        if "gen_ai.prompt.rendered" in a and a["gen_ai.prompt.rendered"].strip().endswith(":"):
            return s, SUSPECTS[2]
    return None, "no clear divergence — inspect sampling"

span, suspect = find_divergence(t1)
print("first divergence point:", span["name"], "  (span", span["span_id"] + ")")
print("implicated suspect    :", suspect)
print("\nevidence on that span:")
for k, v in span["attributes"].items():
    print(f"  {k} = {v}")

**What you just saw.** The `retrieve` span has a `top_score` of **0.21** and three chunks about *shipping*, not refunds — the retrieval served the wrong context. Every span after it (the model dutifully answering from shipping chunks, then the final answer) is *downstream* of that. Had you stopped at the final answer and blamed "the model hallucinated," you'd have tuned the wrong thing. The wrong final answer is a symptom; the **empty/irrelevant retrieval is the cause**.


### A second planted case: tool failure improvised into a "fact"

Same method, different suspect. Here the user asked for an account balance.


In [ ]:
t2 = load_trace("trace-tool-failure-improvised.json")
print_trace(t2)
print()
span2, suspect2 = find_divergence(t2)
print("first divergence point:", span2["name"], "->", suspect2)
for ev in span2.get("events", []):
    print(f"  recorded: {ev['exception.type']}: {ev['exception.message']}")
final = spans_in_order(t2)[-1]
print("\nmodel's improvised answer:", final["attributes"]["gen_ai.completion.text"])
print("...but get_balance ERRORED — the $0.00 balance is invented, not retrieved.")

### Reproduce — at the right granularity

You cannot reliably replay a whole multi-step run: tools have side effects and the world moved (that's why mature setups record tool results — the **fixture pattern** from API testing — so a run replays against *recorded* responses). But you *can* replay a **single model call** exactly, using the rendered prompt and params in the trace, and **resample it 10×** to tell a tail event from the norm.

Below we replay the retrieval-miss case's first model call. In `MOCK=1` we resample a seeded distribution of canned verdicts (deterministic); `MOCK=0` would hit the live model 10× with the *recorded rendered prompt*.


In [ ]:
def grounded_in_context(answer, context):
    """Toy grader: did the answer stick to the provided context?"""
    return "shipping" in answer.lower() or context.strip() == ""

def resample_single_call(trace, span_id, n=10):
    span = next(s for s in trace["spans"] if s["span_id"] == span_id)
    rendered = span["attributes"]["gen_ai.prompt.rendered"]
    if MOCK:
        # Seeded: with garbage context the model usually improvises -> 'ungrounded'.
        pool = (["ungrounded"] * 8) + ["grounded"] * 2
        random.shuffle(pool)
        verdicts = pool[:n]
    else:
        from anthropic import Anthropic
        client = Anthropic()
        verdicts = []
        for _ in range(n):
            r = client.messages.create(model="claude-sonnet-4-6", max_tokens=128,
                                       messages=[{"role": "user", "content": rendered}])
            txt = r.content[0].text if r.content else ""
            verdicts.append("grounded" if "shipping" in txt.lower() else "ungrounded")
    return verdicts

verdicts = resample_single_call(t1, "0x0003", n=10)
ungrounded = verdicts.count("ungrounded")
print("resample verdicts:", verdicts)
print(f"ungrounded {ungrounded}/10 -> this is the NORM, not a tail event: the bug is the retrieval, fix retrieval/prompt")
print("(if it were 1/10, you'd file it as a rare sampling tail and move on.)")

### Fix it in the eval suite, not just the code

The last step of *every* investigation: add the failing case to the **golden set**, so the flywheel guarantees it stays fixed. A debugging session that doesn't end in an eval case is one you'll repeat. We emit a case in the shape Ch 22 (22-03) consumes and append it to a local golden file. (This writes only inside this chapter folder; the real golden set lives with the eval harness.)


In [ ]:
def eval_case_from_trace(trace, divergence_span_id, expected, bug_label):
    s = next(x for x in trace["spans"] if x["span_id"] == divergence_span_id)
    return {
        "id": f"regression-{trace['trace_id'][-4:]}",
        "source": "production-trace",
        "trace_id": trace["trace_id"],
        "bug": bug_label,
        "input": s["attributes"].get("retrieval.query")
                 or s["attributes"].get("gen_ai.prompt.rendered", "")[:120],
        "expected": expected,
        "assert": "answer cites the actual refund policy, not shipping text",
    }

case = eval_case_from_trace(t1, "0x0002", "refund window per policy", "retrieval-miss")
golden = DATA / "golden_regression.jsonl"
with golden.open("a", encoding="utf-8") as f:
    f.write(json.dumps(case) + "\n")
print("added to golden set ->", golden)
print(json.dumps(case, indent=2))
print("\nIn the real flow this lands in blueprints/eval-harness/ (Ch 22) and gates future deploys.")

### Dashboards at two altitudes

Make health visible at two altitudes — and **every chart drills to traces** (a chart you can't drill into is a rumor):
- **Operational** ("is it on fire?"): request rate, error rate, latency percentiles, token spend/hour, tool-failure rate, provider status.
- **Quality** ("are we getting better?"): task-success / eval trends, feedback rate, cost per resolved task, slice scores, top failure buckets.

We compute a small **operational** snapshot from a batch of run records.


In [ ]:
# A small synthetic batch of recent runs (seeded) standing in for a metrics window.
def synth_runs(n=200):
    runs = []
    for _ in range(n):
        err = random.random() < 0.04
        tool_fail = random.random() < 0.07
        latency = random.gauss(2300, 600) if not err else random.gauss(5200, 900)
        runs.append({"latency_ms": max(150, latency), "error": err,
                     "tool_fail": tool_fail, "tokens": random.randint(1500, 3200)})
    return runs

def pctile(xs, p):
    xs = sorted(xs)
    k = min(len(xs) - 1, int(round((p / 100) * (len(xs) - 1))))
    return xs[k]

runs = synth_runs()
lat = [r["latency_ms"] for r in runs]
print("OPERATIONAL dashboard (last window):")
print(f"  requests        : {len(runs)}")
print(f"  error rate      : {sum(r['error'] for r in runs)/len(runs):.1%}")
print(f"  tool-failure    : {sum(r['tool_fail'] for r in runs)/len(runs):.1%}")
print(f"  latency p50/p95 : {pctile(lat,50):.0f} / {pctile(lat,95):.0f} ms")
print(f"  token spend     : {sum(r['tokens'] for r in runs):,} tokens this window")
print("  -> every number drills to the underlying traces in the platform")

### Session-level metrics (the conversational tier the capstone needs)

Most metrics are *run-centric* — one request, one number — but a user doesn't experience runs, they experience a **session**. Aggregate a second tier across the consecutive runs of one conversation, stitched by the `session.id` 23-01 stamps on every trace: **turns-to-resolution**, **conversation success rate**, **abandonment/drop-off**, and **cost-per-resolved-conversation** (the honest unit cost — a session that "succeeds" only after nine expensive turns is not the win a per-run chart implies). This is the observability counterpart to Ch 22's user-simulator evals.


In [ ]:
sessions = json.loads((DATA / "sessions.json").read_text(encoding="utf-8"))["sessions"]

n = len(sessions)
resolved = [s for s in sessions if s["resolved"]]
abandoned = [s for s in sessions if s["abandoned"]]

conv_success = len(resolved) / n
abandonment = len(abandoned) / n
turns_to_resolution = statistics.mean(len(s["runs"]) for s in resolved)
cost_per_resolved = sum(sum(r["cost_usd"] for r in s["runs"]) for s in resolved) / len(resolved)

print("SESSION-level metrics (across", n, "conversations):")
print(f"  conversation success rate     : {conv_success:.0%}")
print(f"  abandonment / drop-off        : {abandonment:.0%}")
print(f"  avg turns-to-resolution       : {turns_to_resolution:.1f}")
print(f"  cost-per-RESOLVED-conversation: ${cost_per_resolved:.4f}")
print("\nNote: a per-run cost chart hides the 4- and 5-turn sessions; only the session unit cost is honest.")

### ⚠️ Pitfall: paging on the wrong signal

Alerting follows the operational/quality split. **Page** a human for operational symptoms — error/latency/cost spikes, provider outages, tool-failure storms (a spend-per-hour alert has saved many teams months). **Don't page** on slow *quality* drift: a two-point eval dip is tomorrow's error-analysis **ticket**, not a 3 a.m. wake-up. Below, a tiny router encodes that discipline.


In [ ]:
def route_alert(signal, value, baseline):
    """Return ('page'|'ticket'|'ok', reason). Operational spikes page; quality drift tickets."""
    operational = {"error_rate", "latency_p95", "cost_per_hour", "tool_failure_rate", "provider_outage"}
    quality = {"eval_score", "feedback_rate", "task_success"}
    if signal in operational:
        if signal == "provider_outage" and value:
            return "page", "provider outage — external behavior change"
        if value >= baseline * 1.5:
            return "page", f"{signal} spiked {value/baseline:.1f}x over baseline"
        return "ok", f"{signal} within range"
    if signal in quality:
        if value <= baseline - 0.02:
            return "ticket", f"{signal} drifted {baseline - value:.3f} below baseline — error-analysis, not a page"
        return "ok", f"{signal} stable"
    return "ok", "unknown signal"

checks = [
    ("error_rate", 0.12, 0.04),
    ("cost_per_hour", 95.0, 40.0),
    ("provider_outage", True, False),
    ("eval_score", 0.83, 0.85),   # 2-pt dip -> ticket, NOT a page
    ("latency_p95", 4900, 4800),  # within range
]
for sig, val, base in checks:
    action, reason = route_alert(sig, val, base)
    print(f"  {action.upper():6} | {sig}: {reason}")

### On-call's new wrinkle, and what a platform adds

Many agentic incidents are **external behavior changes** — a provider model update, a degraded API, a rate-limit shift — that *no repo diff explains*. So the runbook starts differently: **provider status + recent model/config versions first**, then failing traces and the divergence point, then mitigation **levers** you prepared in advance (model fallback, pinned previous prompt version, scope-narrowing feature flag, graceful degradation).

And this is where a **platform** (§23.2) earns its keep beyond raw OTel: trace/session **search** ("every run over \$5", "every trace tagged `refund` that errored yesterday"), one-click **add-to-dataset**, **annotation queues**, online/offline judge scoring, and prompt versions stamped on every trace. The self-host vs SaaS choice is a **data-governance** decision — traces hold real user prompts.


In [ ]:
RUNBOOK = [
    "1. Check provider status page + recent model/config versions (external change first).",
    "2. Pull representative failing traces; walk each to its first divergence point.",
    "3. Apply a mitigation lever: model fallback / pinned previous prompt version /",
    "   scope-narrowing feature flag / graceful degradation to a simpler flow.",
    "4. File the cluster as an eval case; only then resume root-cause work.",
]
print("AGENTIC ON-CALL RUNBOOK")
for line in RUNBOOK:
    print("  " + line)

# What a platform's saved search would pull from our fixtures:
expensive = [s for s in sessions if sum(r["cost_usd"] for r in s["runs"]) > 0.020]
print("\nplatform search 'sessions over $0.020':", [s["session_id"] for s in expensive])

## 🎯 Senior lens

Triage by **distribution, not drama**. A spectacular one-off hallucination pulls the whole team into a rabbit hole while a boring retrieval bug quietly degrades twenty percent of traffic. Ask of every failure: *how often does the cluster it belongs to occur, and what does that cluster cost us?* Trace data answers that; anecdotes never do. Spend the week on the biggest cluster (our retrieval-miss case affects a whole query class), file the dramatic outlier as an eval case, and move on.


## 📋 Observability readiness checklist (§23)

- [ ] Every run is a trace; every model/tool/retrieval call a span with the OTel GenAI attributes.
- [ ] Trace context propagates across queues, workers, and sub-agents — one run, one tree.
- [ ] The *rendered* prompt, sampling params, retrieved chunks (with scores), and raw tool I/O are captured — with truncation, redaction, and a retention policy.
- [ ] Prompt/config versions and the git SHA are stamped on every trace.
- [ ] Cost is computed per run and aggregated per user/agent/feature; anomaly alerts + in-code budget ceilings exist.
- [ ] User feedback and eval verdicts are joined to traces by trace id.
- [ ] Production traces flow to a platform where they can be turned into datasets, scored, and diffed — and self-host-vs-SaaS is a deliberate data-governance choice.
- [ ] Operational and quality dashboards exist; every chart drills down to traces.
- [ ] Pages are reserved for operational symptoms; quality drift routes to the weekly error-analysis ritual.
- [ ] The on-call runbook covers provider incidents and lists mitigation levers.
- [ ] Every debugging session ends by adding the failing case to the eval suite.


## Recap

- Debugging an agent **inverts** the loop: investigate from the trace, don't reproduce-first.
- Walk the span tree to the **first divergence point**; the wrong answer is downstream of the real cause (here: a 0.21-score retrieval miss; and a tool error improvised into a fake balance).
- **Reproduce at the right granularity** — replay a single call and resample 10× to separate a tail event from the norm; whole-run replay needs *recorded* tool results.
- **End in an eval case** — the failing trace becomes a golden regression case (back to 22-03).
- Dashboards at two altitudes; **session-level** metrics (turns-to-resolution, success, abandonment, cost-per-resolved-conversation) are the honest product unit.
- **Page on operational symptoms, ticket on quality drift**; the runbook starts with provider status because many incidents are external behavior changes.


## Exercises

Each exercise *changes* a fixture or threshold and asks you to **predict** the outcome first.

1. **Plant a truncation bug.** Add a third fixture `trace-truncated-context.json` where the rendered prompt is suspiciously short (context silently truncated). Extend `find_divergence` to flag it. *Predict* which suspect fires, then verify.
2. **Flip a tail into the norm.** In `resample_single_call`, change the MOCK pool to 1 ungrounded / 9 grounded. *Predict* the verdict and what you'd do differently (file vs fix), then run it.
3. **Make abandonment page-worthy?** Add three abandoned sessions to `data/sessions.json`. *Predict* the new abandonment rate and argue whether it should *page* or *ticket* under the §23.5 rule, then recompute.
4. **Tighten the alert router.** Change `route_alert` so a `cost_per_hour` over an absolute \$80/hr always pages regardless of baseline. *Predict* which of the sample checks change, then confirm.

In [ ]:
# Exercise 1: add a truncated-context fixture and extend find_divergence.


In [ ]:
# Exercise 2: flip the resample pool to 1/9 and reinterpret.


In [ ]:
# Exercises 3 & 4: add abandoned sessions; add an absolute cost page rule.


## Next

- **Blueprint (production version):** [`../../../blueprints/observability-stack/`](../../../blueprints/observability-stack/) — the OTel tracing + cost-accounting + dashboards stack you just prototyped, with the backend swappable. The eval case you emitted lands in [`../../../blueprints/eval-harness/`](../../../blueprints/eval-harness/) (Ch 22).
- **Capstone:** these *observe*-station habits wire into [`../../../capstone/`](../../../capstone/) `telemetry.py` — closing the flywheel back into Ch 22's eval suite (checkpoint `ch23-observability`). The session/cost signals here become the SLOs formalized in Ch 42.
- **Book:** §23.4 (debugging non-deterministic failures), §23.5 (dashboards, alerting & on-call), §23.2 (what platforms add), §23 readiness checklist.